# IntDist: microwave spectrum of trapped KRb over the trap-intensity distribution

For a chosen set of populated N=0 hyperfine states, simulate the N=0 → N=1 microwave
spectrum, first for a single molecule and then averaged over the distribution of trap
intensities that a thermal cloud samples.

The functions live in `IntDist.jl`; this notebook only sets parameters and plots.

Conventions
- Energies and frequencies in Hz (not angular), times in s, intensities in W/m².
- `MicrowaveDrive(angle_deg, pi_time)`: the field amplitude gives a π-pulse time `pi_time` on a
  0.33 D transition driven by the σ⁻ component of the field.
- Initial-state `weights` are populations.
- Spectra come back as `states × frequencies` matrices; sum over `dims = 1` for the total.

In [ ]:
using LinearAlgebra, Statistics, Random
using PyPlot

include("diatomic.jl")
using .diatomic_jl.MoleculeTypes
using .diatomic_jl
import .diatomic_jl.Hamiltonian: getDipoleMatrix

include("IntDist.jl")
using .IntDist

## 1. Molecules in the trap

Sample positions from a thermal cloud in the loading trap, then evaluate each molecule's
intensity in the spectroscopy trap.

In [ ]:
beams = [GaussianBeam([0, 0, 0], [0.0, 1.0, 0.0], 35e-6),
         GaussianBeam([0, 0, 0], [0.0, -sin(70pi/180), cos(70pi/180)], 100e-6)]

peaks_load  = [5.051656883498337, 2.0133327108188115] * 1e7     # W/m^2, where the cloud thermalises
peaks_probe = [130.16280729349387, 3.3555545180313526] * 1e7    # W/m^2, during spectroscopy
T_mol = 500e-9
n_mol = 10_000

pos, vel = sample_thermal(trap_potential(beams, peaks_load), KRb_mass, T_mol, n_mol;
                          box = [100e-6, 100e-6, 400e-6], rng = MersenneTwister(1))
st = thermal_stats(pos, vel, KRb_mass)
println("Temperature      ", st.temperature * 1e9, " nK")
println("Trap frequencies ", st.trap_freq, " Hz")

In [ ]:
I_probe = trap_intensity(beams, peaks_probe)
intensities = [I_probe(pos[:, k]...) for k in 1:n_mol]
I_mean, I_std = mean(intensities), std(intensities)
println("Intensity ", I_mean * 1e-7, " ± ", I_std * 1e-7, " (1e7 W/m^2)")

clf()
hist(intensities * 1e-7, bins = 60)
xlabel("Intensity (1e7 W/m²)")
ylabel("Molecules")
figure(gcf())

## 2. Molecular structure on an intensity grid

Diagonalise once per grid intensity. Each molecule is later assigned to the nearest grid
point, so the grid spacing (σ/10 here) sets the intensity resolution. The first grid point
is the bare molecule.

In [ ]:
MoleculeOperator = Molecule.generateMolecule(K40Rb87T, 2)
Hmol = Hamiltonian.generateHamiltonian(MoleculeOperator, dirE = [0.0, 1.0, 0.0],
           Beams = Dict("B" => Hamiltonian.OpticalBeam([1.0, 0.0, 0.0], [0, 0.0, 1.0])))  # kVector, polarization
dipOp = getDipoleMatrix(Hmol.MolOp)
ground, excited = manifolds(Hmol)       # N=0 and N=1 eigenstate indices

B_field, E_dc = 50e-4, 47e2
I_grid = [0.0; collect((I_mean - 5I_std):(I_std/10):(I_mean + 5I_std))]
scan = solve.scanIntensity(Hmol, B_field, E_dc, I_grid);

## 3. Spectrum of the bare molecule

Full N≤1 calculation for the two groups of initial states.

In [ ]:
K3 = [2, 10, 18, 26]     # K = -3
K4 = [1, 8, 16, 24]      # K = -4
t_pulse = 200e-6
freqs = collect(2228.08:0.005:2228.4) .* 1e6

bare = DressedMolecule(scan[1], dipOp, MicrowaveDrive(angle_deg = 88.2, pi_time = 1.99e-3), ground, excited)
sK3 = vec(sum(spectrum(bare, freqs, K3, ones(length(K3)), t_pulse), dims = 1))
sK4 = vec(sum(spectrum(bare, freqs, K4, ones(length(K4)), t_pulse), dims = 1))

clf()
plot(freqs * 1e-6, sK3, label = "K = -3")
plot(freqs * 1e-6, sK4, label = "K = -4")
xlabel("Microwave frequency (MHz)")
ylabel("N=1 population")
legend()
figure(gcf())

`spectrum_truncated` keeps only the initial state and the N=1 states within `window` of
resonance. It is what the intensity average uses, so check it against the full calculation
at the drive strength you care about.

In [ ]:
bare2 = DressedMolecule(scan[1], dipOp, MicrowaveDrive(angle_deg = 87.5, pi_time = 199e-6), ground, excited)
freqs_fine = collect(2228.08:0.001:2228.4) .* 1e6
K4b = [8, 16, 24]
full  = vec(sum(spectrum(bare2, freqs_fine, K4b, ones(3), t_pulse), dims = 1))
trunc = vec(sum(spectrum_truncated(bare2, freqs_fine, K4b, ones(3), t_pulse; window = 10e3), dims = 1))

clf()
plot(freqs_fine * 1e-6, full, label = "full N≤1")
plot(freqs_fine * 1e-6, trunc, "--", label = "truncated, 10 kHz window")
xlabel("Microwave frequency (MHz)")
ylabel("N=1 population")
legend()
figure(gcf())

## 4. Spectrum averaged over the intensity distribution

The window is centred on the strongest line from `state` at the mean intensity; set
`freqs_avg` by hand to look elsewhere.

In [ ]:
state = 24
t_avg = 1e-3
drive_avg = MicrowaveDrive(angle_deg = 87.5, pi_time = 398e-6)

m_mean = DressedMolecule(scan[argmin(abs.(I_grid .- I_mean))], dipOp, drive_avg, ground, excited)
f_line = let j = excited[argmax(abs.(m_mean.V[excited, state]))]
    m_mean.E[j] - m_mean.E[state]
end
freqs_avg = collect(range(f_line - 400e3, f_line + 400e3, length = 161))

total, counts = intensity_averaged_spectrum(intensities, scan, dipOp, drive_avg, ground, excited,
                                            freqs_avg, [state], [1.0], t_avg; window = 20e3)
at_mean = vec(spectrum_truncated(m_mean, freqs_avg, [state], [1.0], t_avg; window = 20e3))

clf()
plot(freqs_avg * 1e-6, total / n_mol, label = "averaged over $n_mol molecules")
plot(freqs_avg * 1e-6, at_mean, label = "mean intensity only")
xlabel("Microwave frequency (MHz)")
ylabel("N=1 population per molecule")
legend()
figure(gcf())

## 5. Repeated pulses with partial detection

Each cycle is a microwave pulse followed by a measurement that detects a fraction
`fidelity` of the N=1 population and removes it.

In [ ]:
mw = DressedMolecule(scan[1], dipOp, MicrowaveDrive(angle_deg = 88.2, pi_time = 88.5e-6), ground, excited)
ms_K3 = vec(sum(multistep_spectrum(mw, freqs, K3, ones(4), 10e-6, 200, 0.01), dims = 1))
ms_K4 = vec(sum(multistep_spectrum(mw, freqs, K4, ones(4), 10e-6, 200, 0.01), dims = 1))

clf()
plot(freqs * 1e-6, ms_K3, label = "K = -3, 200 × 10 µs")
plot(freqs * 1e-6, ms_K4, label = "K = -4, 200 × 10 µs")
xlabel("Microwave frequency (MHz)")
ylabel("Accumulated signal")
legend()
figure(gcf())

In [ ]:
mw2 = DressedMolecule(scan[1], dipOp, MicrowaveDrive(angle_deg = 88.2, pi_time = 199e-6), ground, excited)
ms_first = vec(sum(multistep_spectrum(mw2, freqs, K3, ones(4), 5e-6, 300, 0.01; first_pulse = 150e-6), dims = 1))

clf()
plot(freqs * 1e-6, ms_first, label = "K = -3, 150 µs then 300 × 5 µs")
xlabel("Microwave frequency (MHz)")
ylabel("Accumulated signal")
legend()
figure(gcf())

## 6. Reaction products detected outside the dark line

Monte Carlo over products that are born during `t_dark`, fly out at the KRb product speed,
and are probed every `step_time`. Detection is only possible outside the dark line.

In [ ]:
rempi_cfg = REMPIConfig(1.5e-3, 0.1e-3, 0.1)     # beam waist, dark-line half-width (m), peak ionisation probability
mp = DressedMolecule(scan[1], dipOp, MicrowaveDrive(angle_deg = 88.2, pi_time = 89e-6), ground, excited)
prod_opts = (t_dark = 20e-6, microwave_time = 10e-6, step_time = 50e-6, n_steps = 200,
             speed = product_speeds(0, 1).KRb, n_particles = 100)

pK3 = vec(sum(product_spectrum(mp, freqs_fine, K3, ones(4), linear_detection(rempi_cfg); prod_opts...), dims = 1))
pK4 = vec(sum(product_spectrum(mp, freqs_fine, K4, ones(4), linear_detection(rempi_cfg); prod_opts...), dims = 1))

clf()
plot(freqs_fine * 1e-6, pK3, label = "K = -3")
plot(freqs_fine * 1e-6, pK4, label = "K = -4")
xlabel("Microwave frequency (MHz)")
ylabel("Signal per product")
legend()
figure(gcf())

Same simulation with the ionisation yield taken from integrating a three-level ODE over the
REMPI pulse (`REMPILookup.jl`, needs DifferentialEquations).

In [ ]:
include("REMPILookup.jl")
using .REMPILookup

mp2 = DressedMolecule(scan[1], dipOp, MicrowaveDrive(angle_deg = 88.2, pi_time = 88.5e-6), ground, excited)
ode_detection = lookup_detection(REMPIConfig(0.75e-3, 0.1e-3, 0.1))
freqs_ode = collect((2228.224 - 0.15):0.001:(2228.224 + 0.15)) .* 1e6

oK3 = vec(sum(product_spectrum(mp2, freqs_ode, K3, ones(4), ode_detection; prod_opts...), dims = 1))
oK4 = vec(sum(product_spectrum(mp2, freqs_ode, K4, ones(4), ode_detection; prod_opts...), dims = 1))

clf()
plot(freqs_ode * 1e-6, oK3, label = "K = -3")
plot(freqs_ode * 1e-6, oK4, label = "K = -4")
xlabel("Microwave frequency (MHz)")
ylabel("Signal per product")
legend()
figure(gcf())